# EDA — Dataset regenerado v2 vs original

Análisis exploratorio de `data/dataset_regenerado_v2_10000_clientes.csv` (generado en `dataset_regeneration_v2.ipynb`)
y **comparación contra el dataset original** (`data/dataset_dummy_tarjetas_santander_10000_clientes.csv.xlsx`).
Donde ayuda, se incluye también la v1 (`data/dataset_regenerado_10000_clientes.csv`) para ver cuánto se acercó.

La v2 sigue las **políticas v2** (`dataset_regeneration_v2.ipynb`): toma la escala del original (ingreso, gasto, transacciones, logins, e-commerce, promos) y, donde hay un dato real, lo usa en su lugar (Banxico para pagos, uso de la línea, totaleros y niveles de tarjeta; Santander para NPS y satisfacción). **No busca copiar el original**: algunas variables se alejan a propósito porque el original es uniforme y la realidad no.

> ⚠️ Este EDA fue elaborado por una **IA (Claude, de Anthropic)**. Los datos son sintéticos.

**Contenido**
1. Setup y carga
2. Calidad de datos
3. Comparación de distribuciones con el original
4. Segmentos y tarjetas vs original
5. Diferencias entre segmentos (e-commerce y demás)
6. Relaciones y validación de políticas
7. Perfil por segmento y por tarjeta
8. Bolsas de oportunidad
9. Conclusiones

## 1. Setup y carga

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)
pio.renderers.default = "plotly_mimetype"

# Misma paleta y tema que los otros EDA
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GRIS = "#a3a29c"
NEUTRO = "#f0efec"
DIVERGENTE = [[0, "#b83a3a"], [0.25, "#e88a89"], [0.5, NEUTRO], [0.75, "#86b6ef"], [1, "#1c5cab"]]
pio.templates["santander"] = go.layout.Template(layout=dict(
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#0b0b0b"),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", colorway=PAL,
    xaxis=dict(gridcolor="#e8e7e3", zerolinecolor="#c3c2b7", linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e8e7e3", zerolinecolor="#c3c2b7", linecolor="#c3c2b7"),
    hoverlabel=dict(bgcolor="white", font_size=12), margin=dict(l=60, r=30, t=60, b=50), bargap=0.25))
pio.templates.default = "simple_white+santander"

original = pd.read_excel("data/dataset_dummy_tarjetas_santander_10000_clientes.csv.xlsx", sheet_name=0)
v1 = pd.read_csv("data/dataset_regenerado_10000_clientes.csv")
clientes = pd.read_csv("data/dataset_regenerado_v2_10000_clientes.csv")   # v2
DATASETS = {"Original": original, "v1": v1, "v2": clientes}
COLOR_DS = {"Original": GRIS, "v1": PAL[1], "v2": PAL[0]}   # el original es la referencia (gris)

SEGMENTOS = ["Mass", "Select", "Premier"]
COLOR_SEG = dict(zip(SEGMENTOS, PAL))
# En el original, Select (3 %, World Elite) y Premier (23 %, Platinum) parecen intercambiados (ver EDA_segmentos).
# Para comparar se usa el TIER: Mass → Mass, segmento medio → Select, segmento alto → Premier.
TIER_ORIGINAL = {"Mass": "Mass", "Premier": "Select", "Select": "Premier"}
original["tier"] = original.segmento.map(TIER_ORIGINAL)
v1["tier"] = v1.segmento
clientes["tier"] = clientes.segmento

TARJETAS = ["Free", "LikeU", "Aeromexico Blanca", "Fiesta Rewards Oro", "Santander Gold",
            "Santander Platinum", "Fiesta Rewards Platinum", "Aeromexico Platinum", "World Elite"]
NUM = original.select_dtypes("number").columns.tolist()
for n, d in DATASETS.items():
    print(f"{n:9s}", d.shape)

Original  (10000, 17)
v1        (10000, 17)
v2        (10000, 17)


## 2. Calidad de datos (v2)

In [2]:
calidad = pd.DataFrame({"dtype": clientes.dtypes.astype(str), "nulos": clientes.isna().sum(),
                        "unicos": clientes.nunique(), "min": clientes.min(numeric_only=True),
                        "max": clientes.max(numeric_only=True)})
print("Duplicados de fila:", clientes.duplicated().sum(), "| customer_id duplicados:", clientes.customer_id.duplicated().sum())
display(calidad)

# Mismos rangos que el original
rangos = {c: (original[c].min(), original[c].max()) for c in
          ["uso_tarjeta_pct", "ecommerce_pct", "pagos_puntuales_pct", "activacion_promos_pct",
           "abandono_riesgo", "nps", "satisfaccion", "edad", "antiguedad_meses"]}
fuera = {c: int((~clientes[c].between(*r)).sum()) for c, r in rangos.items()}
print("Registros fuera del rango del original:", fuera)

Duplicados de fila: 0 | customer_id duplicados: 0


,dtype,nulos,unicos,min,max
abandono_riesgo,float64,0,726,0.00,0.92
activacion_promos_pct,float64,0,945,0.00,100.00
antiguedad_meses,int64,0,180,1.00,180.00
app_logins_mes,int64,0,28,1.00,32.00
customer_id,str,0,10000,NaN,NaN
ecommerce_pct,float64,0,977,0.00,100.00
edad,int64,0,58,18.00,75.00
gasto_mensual,float64,0,9976,"1,007.27","81,959.97"
ingreso_mensual,float64,0,9991,"3,231.99","160,264.18"
nps,int64,0,11,0.00,10.00


Registros fuera del rango del original: {'uso_tarjeta_pct': 0, 'ecommerce_pct': 0, 'pagos_puntuales_pct': 0, 'activacion_promos_pct': 0, 'abandono_riesgo': 0, 'nps': 0, 'satisfaccion': 0, 'edad': 0, 'antiguedad_meses': 0}


## 3. Comparación de distribuciones con el original

Para cada variable numérica se compara la media, la dispersión y los percentiles, y se calcula el **estadístico KS**
(distancia máxima entre las dos distribuciones acumuladas: 0 = idénticas, 1 = sin traslape). Como referencia práctica:
**KS < 0.10** es muy cerca, **0.10–0.20** es cerca y **> 0.30** es lejos.

In [3]:
def ks(a, b):
    """Estadístico Kolmogorov–Smirnov de dos muestras (sin scipy)."""
    a, b = np.sort(np.asarray(a, float)), np.sort(np.asarray(b, float))
    x = np.concatenate([a, b])
    return float(np.max(np.abs(np.searchsorted(a, x, "right") / len(a) - np.searchsorted(b, x, "right") / len(b))))

filas = []
for v in NUM:
    o, r = original[v], clientes[v]
    filas.append({"variable": v,
                  "media orig": o.mean(), "media v2": r.mean(), "σ orig": o.std(), "σ v2": r.std(),
                  "p5 orig": o.quantile(.05), "p5 v2": r.quantile(.05),
                  "p50 orig": o.median(), "p50 v2": r.median(),
                  "p95 orig": o.quantile(.95), "p95 v2": r.quantile(.95),
                  "KS v1": ks(o, v1[v]), "KS v2": ks(o, r)})
comp = pd.DataFrame(filas).set_index("variable")
comp["veredicto v2"] = pd.cut(comp["KS v2"], [-1, .10, .20, .30, 1], labels=["muy cerca", "cerca", "moderado", "lejos"])
display(comp)
print(f"KS promedio → v1: {comp['KS v1'].mean():.3f} | v2: {comp['KS v2'].mean():.3f}")

,media orig,media v2,σ orig,σ v2,p5 orig,p5 v2,p50 orig,p50 v2,p95 orig,p95 v2,KS v1,KS v2,veredicto v2
variable,,,,,,,,,,,,,
edad,46.48,41.25,16.73,15.43,20.00,20.00,47.00,40.00,72.00,69.00,0.14,0.14,cerca
antiguedad_meses,91.66,73.76,51.59,52.97,10.00,4.00,92.00,65.00,172.00,167.00,0.34,0.15,cerca
ingreso_mensual,"30,627.46","30,623.67","16,263.78","16,047.70","11,941.93","11,714.54","27,140.17","27,138.31","61,477.81","61,517.87",0.59,0.01,muy cerca
uso_tarjeta_pct,49.95,35.11,28.56,23.75,5.20,4.05,49.85,30.46,94.50,79.74,0.05,0.23,moderado
transacciones_mes,17.98,17.82,4.30,5.22,11.00,10.00,18.00,17.00,25.00,27.00,0.73,0.07,muy cerca
gasto_mensual,"12,049.73","11,911.12","6,052.09","7,080.10","4,118.49","4,014.09","11,115.26","10,217.69","23,483.93","25,530.19",0.16,0.07,muy cerca
ecommerce_pct,49.65,49.65,28.75,22.72,5.40,11.30,49.60,50.00,95.10,88.10,0.19,0.11,cerca
pagos_puntuales_pct,85.00,95.62,8.65,6.62,71.60,79.40,85.00,98.30,98.50,99.90,0.32,0.60,lejos
app_logins_mes,10.03,9.73,3.18,3.75,5.00,4.00,10.00,9.00,16.00,16.00,0.39,0.08,muy cerca


KS promedio → v1: 0.365 | v2: 0.227


In [4]:
orden = comp.sort_values("KS v2").index
fig = go.Figure()
for n in ["v1", "v2"]:
    fig.add_scatter(y=orden, x=comp.loc[orden, f"KS {n}"], mode="markers", name=n,
                    marker=dict(color=COLOR_DS[n], size=11, line=dict(color="#fcfcfb", width=2)),
                    hovertemplate="%{y}<br>KS " + n + ": %{x:.3f}<extra></extra>")
for y in orden:   # conector v1 → v2
    fig.add_shape(type="line", x0=comp.loc[y, "KS v1"], x1=comp.loc[y, "KS v2"], y0=y, y1=y,
                  line=dict(color=GRIS, width=1.5), layer="below")
for x, t in [(0.10, "muy cerca"), (0.20, "cerca")]:
    fig.add_vline(x=x, line=dict(color="#52514e", dash="dot", width=1))
    fig.add_annotation(x=x, y=1.02, yref="paper", text=t, showarrow=False, xanchor="right", font=dict(size=11, color="#52514e"))
fig.update_layout(height=480, title="Distancia al original por variable (KS, menor es mejor)", xaxis_title="estadístico KS",
                  xaxis_range=[0, 1], legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig.show()

In [5]:
fig = make_subplots(rows=4, cols=4, subplot_titles=NUM, vertical_spacing=0.08, horizontal_spacing=0.05)
for i, v in enumerate(NUM):
    lo, hi = np.percentile(np.r_[original[v], clientes[v]], [0, 99.5])
    bins = dict(start=lo, end=hi, size=(hi - lo) / 30)
    for n in ["Original", "v2"]:
        fig.add_histogram(x=DATASETS[n][v], xbins=bins, histnorm="percent", name=n, legendgroup=n, showlegend=(i == 0),
                          marker_color=COLOR_DS[n], opacity=0.6,
                          hovertemplate=v + ": %{x}<br>%{y:.1f}% de clientes<extra>" + n + "</extra>",
                          row=i // 4 + 1, col=i % 4 + 1)
fig.update_layout(height=950, barmode="overlay", bargap=0.02, title="Distribución de cada variable: original (gris) vs v2 (azul), % de clientes",
                  legend=dict(orientation="h", y=1.05, x=1, xanchor="right"))
fig.show()

In [6]:
# Pirámide de edad: v2 y original vs pesos de la política v2
POBLACION = pd.Series([15_571_098, 21_105_726, 18_729_814, 15_640_829, 12_087_127, 7_384_543, 270_410],
                      index=["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75"])
CORTES_EDAD = [17, 24, 34, 44, 54, 64, 74, 75]
pct_rango = lambda d: pd.cut(d.edad, CORTES_EDAD, labels=POBLACION.index).value_counts(normalize=True).sort_index() * 100
edades = pd.DataFrame({"original %": pct_rango(original), "v2 %": pct_rango(clientes),
                       "política %": POBLACION / POBLACION.sum() * 100})
display(edades.round(1))

fig = go.Figure()
for n in ["Original", "v2"]:
    fig.add_bar(x=edades.index, y=edades[f"{n.lower()} %"], name=n, marker_color=COLOR_DS[n],
                hovertemplate="%{x}: %{y:.1f}%<extra>" + n + "</extra>")
fig.add_scatter(x=edades.index, y=edades["política %"], name="Política (pirámide 2026)", mode="markers",
                marker=dict(color="#0b0b0b", size=10, symbol="diamond"), hovertemplate="%{x}: %{y:.1f}%<extra>Política</extra>")
fig.update_layout(height=380, barmode="group", title="Edad por grupo: la v2 replica la pirámide de la política",
                  yaxis_title="% de clientes", legend=dict(orientation="h", y=1.12, x=1, xanchor="right"))
fig.show()

,original %,v2 %,política %
18-24,12.10,16.50,17.20
25-34,17.20,23.40,23.20
35-44,17.10,20.50,20.60
45-54,17.30,17.40,17.20
55-64,17.40,13.10,13.30
65-74,17.20,8.80,8.10
75,1.70,0.40,0.30


**Lectura — distribuciones vs original**

- **El KS promedio contra el original baja de 0.37 (v1) a 0.23 (v2).** El promedio esconde dos grupos muy distintos.
- **Cerca del original (KS ≤ 0.16), 8 variables**, donde la política toma la escala del original:
  - ingreso (0.01: misma media, mediana, p5 y p95)
  - gasto, transacciones y logins (≤ 0.08)
  - e-commerce, edad, antigüedad y activación de promos (0.11–0.16)
- **Lejos del original a propósito, 5 variables**, porque la política usa datos reales en lugar de la forma uniforme del original:
  - `pagos_puntuales_pct` (KS 0.60): mediana 98 % vs 85 %. Con los totaleros (≈ 69 %, Banxico), casi todos pagan a tiempo; solo el 5.3 % está por debajo de 80 %.
  - `abandono_riesgo` (KS 0.53): media 0.17 vs 0.50. La mayoría tiene riesgo bajo y una cola tiene riesgo alto, en lugar de una uniforme.
  - `satisfaccion` (0.47) y `nps` (0.33): medias del informe de Santander (4 y 6.9) en lugar de uniformes.
  - `uso_tarjeta_pct` (0.23): media 35 %, como el saldo / límite de Banxico (≈ 36 %).
- Edad y antigüedad siguen algo por debajo del original (mediana 40 vs 47 años y 65 vs 92 meses), por la pirámide poblacional y el tope por edad.
- **Pirámide de edad:** la v2 reproduce los pesos de la política con menos de 1 pp de diferencia en cada grupo. Frente al original, que es uniforme, hay más jóvenes (18–34: 40 % vs 29 %) y menos adultos mayores (65–74: 8.8 % vs 17.2 %).

## 4. Segmentos y tarjetas vs original

In [7]:
mezcla_seg = pd.DataFrame({n: d.tier.value_counts(normalize=True).reindex(SEGMENTOS) * 100 for n, d in DATASETS.items()})
mezcla_seg.index.name = "tier (segmento)"
mezcla_tar = pd.DataFrame({n: d.tarjeta_santander.value_counts(normalize=True).reindex(TARJETAS).fillna(0) * 100
                           for n, d in DATASETS.items()})
display(mezcla_seg.round(1)); display(mezcla_tar.round(1))

fig = make_subplots(rows=1, cols=2, column_widths=[0.3, 0.7], horizontal_spacing=0.12,
                    subplot_titles=["Segmento (por tier), % de clientes", "Tarjeta, % de clientes"])
for n in DATASETS:
    fig.add_bar(y=mezcla_seg.index, x=mezcla_seg[n], orientation="h", name=n, legendgroup=n, marker_color=COLOR_DS[n],
                hovertemplate="%{y}: %{x:.1f}%<extra>" + n + "</extra>", row=1, col=1)
    fig.add_bar(y=mezcla_tar.index, x=mezcla_tar[n], orientation="h", name=n, legendgroup=n, showlegend=False,
                marker_color=COLOR_DS[n], hovertemplate="%{y}: %{x:.1f}%<extra>" + n + "</extra>", row=1, col=2)
fig.update_yaxes(autorange="reversed")
fig.update_layout(barmode="group", height=520, bargap=0.2, bargroupgap=0.08,
                  legend=dict(orientation="h", y=1.12, x=1, xanchor="right"))
fig.show()

display(pd.crosstab(clientes.tarjeta_santander, clientes.segmento).reindex(index=TARJETAS, columns=SEGMENTOS))

# Nivel de tarjeta (agrupación de la política) vs Banxico, Cuadro 1
NIVEL = {"Free": "Clásica", "LikeU": "Clásica", "Aeromexico Blanca": "Oro", "Fiesta Rewards Oro": "Oro",
         "Santander Gold": "Oro", "Santander Platinum": "Platino", "Fiesta Rewards Platinum": "Platino",
         "Aeromexico Platinum": "Platino", "World Elite": "Platino"}
niveles = pd.DataFrame({n: d.tarjeta_santander.map(NIVEL).value_counts(normalize=True) * 100 for n, d in DATASETS.items()})
niveles["Banxico"] = pd.Series({"Clásica": 59.0, "Oro": 31.1, "Platino": 9.8})
display(niveles.reindex(["Clásica", "Oro", "Platino"]).round(1).rename_axis("% de tarjetas por nivel"))

,Original,v1,v2
tier (segmento),,,
Mass,73.70,48.30,78.80
Select,23.20,41.80,14.20
Premier,3.00,10.00,7.00


,Original,v1,v2
tarjeta_santander,,,
Free,20.00,24.50,25.20
LikeU,25.10,15.20,33.90
Aeromexico Blanca,11.80,4.30,12.40
Fiesta Rewards Oro,6.90,4.30,7.30
Santander Gold,10.00,32.50,11.30
Santander Platinum,9.90,6.20,4.00
Fiesta Rewards Platinum,5.20,6.40,1.90
Aeromexico Platinum,8.10,5.90,2.80
World Elite,3.00,0.80,1.30


segmento,Mass,Select,Premier
tarjeta_santander,,,
Free,2520,0,0
LikeU,3389,0,0
Aeromexico Blanca,1243,0,0
Fiesta Rewards Oro,728,0,0
Santander Gold,0,1126,0
Santander Platinum,0,130,267
Fiesta Rewards Platinum,0,69,125
Aeromexico Platinum,0,90,186
World Elite,0,0,127


,Original,v1,v2,Banxico
% de tarjetas por nivel,,,,
Clásica,45.10,39.70,59.10,59.00
Oro,28.60,41.00,31.00,31.10
Platino,26.30,19.30,9.90,9.80


In [8]:
ct = pd.crosstab(clientes.tarjeta_santander, clientes.segmento).reindex(index=TARJETAS, columns=SEGMENTOS)
fig = go.Figure()
for s in SEGMENTOS:
    fig.add_bar(y=ct.index, x=ct[s], name=s, orientation="h", marker_color=COLOR_SEG[s],
                marker_line=dict(color="#fcfcfb", width=2),
                hovertemplate="%{y} · " + s + "<br>%{x:,} clientes<extra></extra>")
fig.update_layout(barmode="stack", height=420, yaxis_autorange="reversed", xaxis_title="clientes",
                  title="v2: clientes por tarjeta (de menor a mayor tier) y segmento",
                  legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig.show()

**Lectura — segmentos y tarjetas**

- **Los niveles de tarjeta coinciden con Banxico:** Clásica 59.1 %, Oro 31.0 % y Platino 9.9 % (Banxico 59.0 / 31.1 / 9.8). El original tenía 45 / 29 / 26, con demasiadas Platino.
- **Segmentos:** Mass 78.8 %, Select 14.2 % y Premier 7.0 %. Mass sigue dominando como en el original (73.7 %), con un Select más chico y un Premier más grande que los del original por tier.
- **Tarjetas de Mass:** Free, Blanca y Oro quedan a ±5 pp del original, y Blanca/Oro respetan el 63/37 del original. LikeU queda más alta (33.9 % vs 25.1 %).
- **Platinum y World Elite** quedan por debajo del original (8.7 % vs 23.2 % las tres Platinum; 1.3 % vs 3 % World Elite). Es la consecuencia directa de calibrar contra Banxico, donde Platino es solo 9.8 %. Entre las Platinum se respeta el 43/22/35 del original.
- En el gráfico apilado, cada tarjeta pertenece a un solo segmento, salvo las tres **Platinum**, que se reparten entre Select (≈ 1/3) y Premier (≈ 2/3) como pide la política.

## 5. Diferencias entre segmentos

En el original, los tres segmentos tienen prácticamente la misma distribución en todo (las variables son independientes).
En v1, al definir el segmento por gasto, todo lo que dependía del gasto se separaba mucho (e-commerce: 39 % / 62 % / 97 %).
Aquí se mide esa separación con la **brecha estandarizada**: (media del tier más alto − media del más bajo) / σ del original.
Una brecha < 0.2 σ es pequeña, y > 0.8 σ es enorme.

In [9]:
VARS_SEG = [v for v in NUM if v not in ("ingreso_mensual", "gasto_mensual", "antiguedad_meses")]  # las 3 definen el segmento
brecha = pd.DataFrame({n: (lambda m: (m.max() - m.min()) / original[VARS_SEG].std())(d.groupby("tier")[VARS_SEG].mean())
                       for n, d in DATASETS.items()})
display(brecha.sort_values("v2", ascending=False).round(2).rename_axis("brecha entre segmentos (en σ del original)"))

orden = brecha.sort_values("v1").index
fig = go.Figure()
for n in DATASETS:
    fig.add_bar(y=orden, x=brecha.loc[orden, n], orientation="h", name=n, marker_color=COLOR_DS[n],
                hovertemplate="%{y}<br>brecha: %{x:.2f} σ<extra>" + n + "</extra>")
fig.add_vline(x=0.2, line=dict(color="#52514e", dash="dot", width=1))
fig.update_layout(barmode="group", height=520, bargap=0.2, bargroupgap=0.08, xaxis_title="brecha máx. entre segmentos (σ del original)",
                  title="Qué tanto se separan los segmentos en cada variable (línea = 0.2 σ)",
                  legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
fig.show()

,Original,v1,v2
brecha entre segmentos (en σ del original),,,
transacciones_mes,0.06,27.31,1.72
app_logins_mes,0.07,9.20,0.89
ecommerce_pct,0.03,2.02,0.75
edad,0.03,0.37,0.41
abandono_riesgo,0.07,0.33,0.23
activacion_promos_pct,0.07,0.68,0.23
uso_tarjeta_pct,0.08,2.21,0.12
satisfaccion,0.06,0.02,0.05
nps,0.03,0.02,0.03


In [10]:
# E-commerce por segmento: media y distribución
ec = pd.DataFrame({n: d.groupby("tier").ecommerce_pct.mean().reindex(SEGMENTOS) for n, d in DATASETS.items()})
display(ec.round(1).rename_axis("e-commerce % medio"))

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=list(DATASETS), horizontal_spacing=0.03)
for j, (n, d) in enumerate(DATASETS.items(), start=1):
    for s in SEGMENTOS:
        fig.add_box(y=d.loc[d.tier == s, "ecommerce_pct"], name=s, marker_color=COLOR_SEG[s], boxmean=True,
                    legendgroup=s, showlegend=(j == 1), row=1, col=j)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(title_text="ecommerce_pct", row=1, col=1)
fig.update_layout(height=420, title="E-commerce % por segmento (◇ = media): original, v1 y v2",
                  legend=dict(orientation="h", y=1.15, x=1, xanchor="right"))
fig.show()

PCT = ["uso_tarjeta_pct", "ecommerce_pct", "pagos_puntuales_pct", "activacion_promos_pct", "abandono_riesgo",
       "transacciones_mes", "app_logins_mes"]
perfil = pd.concat({n: d.groupby("tier")[PCT].mean().reindex(SEGMENTOS).T for n, d in DATASETS.items()}, axis=1)
display(perfil.round(2).rename_axis("media por segmento"))

,Original,v1,v2
e-commerce % medio,,,
Mass,49.60,38.90,46.20
Select,49.90,61.90,59.80
Premier,49.00,97.00,67.70


Original                   v1                   v2               
tier                      Mass Select Premier  Mass Select Premier  Mass Select Premier
media por segmento                                                                     
uso_tarjeta_pct          49.82  50.10   52.02 28.80  60.77   91.79 34.64  36.31   37.95
ecommerce_pct            49.61  49.86   48.98 38.86  61.95   97.02 46.22  59.77   67.73
pagos_puntuales_pct      85.00  84.97   85.19 79.84  77.65   76.13 95.59  95.63   95.87
activacion_promos_pct    50.65  49.60   51.76 44.44  34.53   24.80 51.99  47.87   45.32
abandono_riesgo           0.50   0.50    0.48  0.25   0.18    0.15  0.18   0.15    0.12
transacciones_mes        18.00  17.91   18.15 29.79  64.90  147.13 16.66  21.16   24.07
app_logins_mes           10.05   9.98    9.83  9.84  19.15   39.11  9.26  11.16   12.10

**Lectura — diferencias entre segmentos**

- **Ya no hay variables que separen los segmentos de forma absurda.** En v1 había brechas de 2 a 27 σ; en v2 la mayor es de 1.7 σ.
- **Iguales entre segmentos** (brecha ≤ 0.25 σ): uso de la línea (35 / 36 / 38 %), pagos puntuales (96 % en los tres), satisfacción, NPS, abandono y promociones. Para el uso de la línea es lo que pide la política: según Banxico, las tarjetas altas tienen límites más altos, así que el uso no sube con el segmento.
- ⚠️ **E-commerce vuelve a separarse más:** 46 / 60 / 68 % (Mass / Select / Premier), una brecha de 0.75 σ. Sigue muy lejos de v1 (39 / 62 / 97 %), pero es más que la v2 anterior (49 / 53 / 57 %). Viene del término `12·z(ln gasto)`: el gasto de Premier está ~2 σ arriba de la media, lo que suma ~24 puntos. Si se quiere más plano, bajar el 12 a 5–6 deja la brecha en ~10 puntos.
- **Transacciones** (16.7 → 24.1, 1.7 σ) y **logins** (9.3 → 12.1, 0.9 σ) también suben con el segmento. Es esperable: la política los hace proporcionales al gasto, y el segmento se define por gasto.

## 6. Relaciones y validación de políticas

El original no tiene correlaciones (|ρ| ≤ 0.03); la v2 debe tener las que pide la política.

In [11]:
corr = clientes[NUM].rank().corr()   # Spearman sin scipy
fig = px.imshow(corr, color_continuous_scale=DIVERGENTE, zmin=-1, zmax=1, text_auto=".2f", aspect="auto",
                title="Correlación de Spearman entre variables (v2)")
fig.update_layout(height=620, coloraxis_colorbar=dict(title="ρ"))
fig.show()
fuera_diag = lambda d: d[NUM].rank().corr().where(~np.eye(len(NUM), dtype=bool)).abs().max().max()
print({n: round(fuera_diag(d), 2) for n, d in DATASETS.items()}, "← máx |ρ| fuera de la diagonal")

{'Original': np.float64(0.02), 'v1': np.float64(0.98), 'v2': np.float64(0.89)} ← máx |ρ| fuera de la diagonal


In [12]:
# La regla de segmento (árbol 1) vista en los datos: gasto vs antigüedad
muestra = clientes.sample(3_000, random_state=0)
fig = go.Figure()
for s in SEGMENTOS:
    d = muestra[muestra.segmento == s]
    fig.add_scatter(x=d.antiguedad_meses, y=d.gasto_mensual, mode="markers", name=s,
                    marker=dict(color=COLOR_SEG[s], size=8, opacity=0.6, line=dict(color="#fcfcfb", width=1)),
                    customdata=d.customer_id, hovertemplate="%{customdata}<br>antigüedad %{x} m<br>gasto $%{y:,.0f}<extra>" + s + "</extra>")
for y, x, t in [(15_500, 12, "Select: gasto ≥ 15.5k y antigüedad ≥ 12 m"), (22_000, 24, "Premier: gasto ≥ 22k y antigüedad ≥ 24 m")]:
    fig.add_shape(type="line", x0=x, x1=180, y0=y, y1=y, line=dict(color="#52514e", dash="dot", width=1.5))
    fig.add_shape(type="line", x0=x, x1=x, y0=y, y1=100_000, line=dict(color="#52514e", dash="dot", width=1.5))
    fig.add_annotation(x=np.log10(180), y=np.log10(y), text=t, showarrow=False, xanchor="right", yanchor="bottom",
                       font=dict(size=11, color="#52514e"))
fig.update_xaxes(type="log", title="antigüedad (meses, log)")
fig.update_yaxes(type="log", title="gasto mensual (MXN, log)")
fig.update_layout(height=520, title="Gasto vs antigüedad por segmento (muestra de 3,000; líneas = reglas del árbol)",
                  legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
fig.show()

In [13]:
# Relaciones de la política que no son lineales: e-commerce por edad, abandono por antigüedad
grupos = pd.DataFrame({
    "edad": pd.cut(clientes.edad, CORTES_EDAD, labels=POBLACION.index),
    "antig": pd.cut(clientes.antiguedad_meses, [0, 6, 12, 24, 36, 120, 180],
                    labels=["≤6", "7-12", "13-24", "25-36", "37-120", ">120"]),
})
fig = make_subplots(rows=1, cols=2, subplot_titles=["E-commerce % (mediana) por edad", "Riesgo de abandono (mediana) por antigüedad (meses)"])
for col, (g, v) in enumerate([("edad", "ecommerce_pct"), ("antig", "abandono_riesgo")], start=1):
    m = clientes.groupby(grupos[g], observed=True)[v].median()
    fig.add_scatter(x=m.index.astype(str), y=m.values, mode="lines+markers", line=dict(color=PAL[0], width=2),
                    marker=dict(size=8), showlegend=False, hovertemplate="%{x}<br>%{y:.2f}<extra></extra>", row=1, col=col)
fig.update_layout(height=360)
fig.show()

In [14]:
c = clientes
seg_regla = np.select([(c.gasto_mensual >= 22_000) & (c.antiguedad_meses >= 24),
                       (c.gasto_mensual >= 15_500) & (c.antiguedad_meses >= 12)], ["Premier", "Select"], "Mass")  # U de v2
reglas = pd.Series({
    "edad en [18, 75]": c.edad.between(18, 75).mean(),
    "antigüedad ≤ min((edad−18)·12 + 1, 180)": (c.antiguedad_meses <= np.minimum((c.edad - 18) * 12 + 1, 180)).mean(),
    "gasto ≤ 1.1 × ingreso": (c.gasto_mensual <= 1.1 * c.ingreso_mensual + 0.01).mean(),
    "uso de tarjeta en (0, 100)": c.uso_tarjeta_pct.between(0, 100, inclusive="neither").mean(),
    "segmento = regla del árbol 1": (c.segmento == seg_regla).mean(),
}, name="% de clientes que cumple") * 100
display(reglas.to_frame())

ESPERADO = [  # (variable, contra(dataset), nombre, signo esperado según la política)
    ("gasto_mensual", lambda d: d.ingreso_mensual, "ingreso", +1),
    ("pagos_puntuales_pct", lambda d: (d.ingreso_mensual - d.gasto_mensual) / d.ingreso_mensual, "holgura (ingreso−gasto)/ingreso", +1),
    ("ecommerce_pct", lambda d: d.gasto_mensual, "gasto", +1),
    ("ecommerce_pct", lambda d: d.edad, "edad", -1),
    ("transacciones_mes", lambda d: d.gasto_mensual, "gasto", +1),
    ("app_logins_mes", lambda d: d.ecommerce_pct, "ecommerce", +1),
    ("app_logins_mes", lambda d: d.transacciones_mes, "transacciones", +1),
    ("abandono_riesgo", lambda d: d.satisfaccion, "satisfacción", -1),
    ("abandono_riesgo", lambda d: d.nps, "nps", -1),
    ("abandono_riesgo", lambda d: d.antiguedad_meses.clip(upper=36), "antigüedad (tope 36 m)", -1),
    ("nps", lambda d: d.satisfaccion, "satisfacción", +1),
    ("abandono_riesgo", lambda d: d.pagos_puntuales_pct, "pagos puntuales", -1),
    ("uso_tarjeta_pct", lambda d: d.transacciones_mes, "transacciones", +1),
]
rho = lambda d, v, contra: d[v].rank().corr(contra(d).rank())   # Spearman sin scipy
signos = pd.DataFrame([{"variable": v, "contra": nombre, "esperado": "+" if s > 0 else "−",
                        "ρ v2": rho(c, v, contra), "ρ original": rho(original, v, contra)}
                       for v, contra, nombre, s in ESPERADO])
for seg in SEGMENTOS:
    d = c[c.segmento == seg]
    signos.loc[len(signos)] = ["activacion_promos_pct", f"ingreso (dentro de {seg})", "−",
                               d.activacion_promos_pct.rank().corr(d.ingreso_mensual.rank()), np.nan]
signos["cumple"] = np.where(np.sign(signos["ρ v2"]) == signos.esperado.map({"+": 1, "−": -1}), "✓", "✗")
display(signos)
# Se reporta en vez de detener el notebook: un ✗ es un hallazgo sobre la política (ver lectura).
print("Relaciones que NO tienen el signo esperado:", signos.loc[signos.cumple == "✗", ["variable", "contra"]].values.tolist())

,% de clientes que cumple
"edad en [18, 75]",100.00
"antigüedad ≤ min((edad−18)·12 + 1, 180)",100.00
gasto ≤ 1.1 × ingreso,100.00
"uso de tarjeta en (0, 100)",100.00
segmento = regla del árbol 1,100.00


,variable,contra,esperado,ρ v2,ρ original,cumple
0,gasto_mensual,ingreso,+,0.89,0.02,✓
1,pagos_puntuales_pct,holgura (ingreso−gasto)/ingreso,+,0.15,0.00,✓
2,ecommerce_pct,gasto,+,0.41,0.01,✓
3,ecommerce_pct,edad,−,-0.22,-0.01,✓
4,transacciones_mes,gasto,+,0.56,0.01,✓
5,app_logins_mes,ecommerce,+,0.44,0.01,✓
6,app_logins_mes,transacciones,+,0.40,-0.00,✓
7,abandono_riesgo,satisfacción,−,-0.33,-0.02,✓
8,abandono_riesgo,nps,−,-0.36,-0.00,✓
9,abandono_riesgo,antigüedad (tope 36 m),−,-0.48,0.02,✓


Relaciones que NO tienen el signo esperado: [['activacion_promos_pct', 'ingreso (dentro de Mass)'], ['activacion_promos_pct', 'ingreso (dentro de Select)']]


**Lectura — relaciones y validación**

- Las 5 reglas duras se cumplen al **100 %**, incluida la reconstrucción del segmento con los umbrales v2.
- **14 de las 16 relaciones** tienen el signo que pide la política (la de promociones en Premier pasa por poco, ρ = −0.01). Las más fuertes son:
  - gasto ~ ingreso (ρ = 0.89)
  - NPS ~ satisfacción (0.71)
  - transacciones ~ gasto (0.56)
  - abandono ~ antigüedad (−0.48)
- Ojo con **gasto ~ ingreso (ρ = 0.89)**: con σ_log de 0.25, el gasto es casi un múltiplo fijo del ingreso, y para modelar son casi la misma variable.
- ⚠️ **No se cumple:** la activación de promociones **no baja con el ingreso dentro de cada segmento** (ρ ≈ 0, con signo positivo en Mass y Select). No es un error del código: dos reglas de la política se anulan.
  - El ingreso resta poco: −5 pts por cada factor *e*, y solo por encima de la media del segmento.
  - Al mismo tiempo, el ingreso alto sube la probabilidad de ser totalero, y ser totalero suma +15 pts de diferencia en promociones.
  - Con `−10·ln(ingreso / media del segmento)`, sin el `máx(0, ·)`, la relación sale negativa en los tres segmentos y la media se queda en ≈ 51 %.
- **Gasto vs antigüedad:** se ven los rectángulos exactos del árbol 1. No hay ruido en la frontera: un cliente con 15,499 MXN de gasto es Mass y uno con 15,500 es Select.
- **E-commerce por edad:** la mediana es estable hasta los 44 años (52–56 %) y luego cae (43 % a los 55–64, 34 % a los 65–74), como pide la política.
- **Abandono por antigüedad:** el riesgo mediano cae de 0.35 en los primeros 6 meses a 0.12 a los 3 años y luego se aplana (0.09), por el tope de 36 meses en la fórmula de lealtad. **Los primeros 12 meses son la ventana crítica.**

## 7. Perfil por segmento y por tarjeta (v2)

In [15]:
VARS_PERFIL = ["edad", "antiguedad_meses", "ingreso_mensual", "gasto_mensual", "uso_tarjeta_pct", "transacciones_mes",
               "ecommerce_pct", "pagos_puntuales_pct", "app_logins_mes", "nps", "satisfaccion",
               "abandono_riesgo", "activacion_promos_pct"]
display(clientes.groupby("segmento")[VARS_PERFIL].median().reindex(SEGMENTOS).T.rename_axis("mediana"))

CLAVE = ["ingreso_mensual", "gasto_mensual", "uso_tarjeta_pct", "ecommerce_pct", "abandono_riesgo", "activacion_promos_pct"]
fig = make_subplots(rows=1, cols=len(CLAVE), subplot_titles=CLAVE, horizontal_spacing=0.04)
for i, v in enumerate(CLAVE, start=1):
    for s in SEGMENTOS:
        fig.add_box(y=clientes.loc[clientes.segmento == s, v], name=s, marker_color=COLOR_SEG[s], boxmean=True,
                    legendgroup=s, showlegend=(i == 1), row=1, col=i)
fig.update_xaxes(showticklabels=False)
fig.update_layout(height=400, title="Métricas clave por segmento (◇ = media)",
                  legend=dict(orientation="h", y=1.18, x=1, xanchor="right"))
fig.show()

segmento,Mass,Select,Premier
mediana,,,
edad,38.00,44.00,47.00
antiguedad_meses,58.00,83.00,98.00
ingreso_mensual,"23,840.85","43,571.38","60,835.10"
gasto_mensual,"8,782.85","18,009.12","26,528.49"
uso_tarjeta_pct,30.18,30.54,33.37
transacciones_mes,16.00,21.00,24.00
ecommerce_pct,46.40,60.00,67.60
pagos_puntuales_pct,98.30,98.40,98.40
app_logins_mes,9.00,11.00,12.00


In [16]:
med = clientes.groupby("tarjeta_santander")[VARS_PERFIL].median().reindex(TARJETAS)
z = (med - med.mean()) / med.std()   # color = qué tan arriba/abajo está la tarjeta vs las demás
fig = go.Figure(go.Heatmap(z=z.values, x=VARS_PERFIL, y=TARJETAS, colorscale=DIVERGENTE, zmid=0, zmin=-2.5, zmax=2.5,
                           text=med.map(lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:.2f}").values,
                           texttemplate="%{text}", textfont=dict(size=10),
                           hovertemplate="%{y}<br>%{x}: %{text}<extra></extra>", colorbar=dict(title="z")))
fig.update_layout(height=480, yaxis_autorange="reversed", xaxis_tickangle=-35,
                  title="Mediana por tarjeta (texto) y posición relativa entre tarjetas (color, z-score)")
fig.show()

**Lectura — perfiles**

- **Por segmento** (medianas), el ingreso sube de 23.8k (Mass) a 43.6k (Select) y 60.8k (Premier); el gasto, de 8.8k a 18.0k y 26.5k. Premier también es mayor (47 vs 38 años) y más antiguo (98 vs 58 meses).
- **Lo que no cambia entre segmentos** (por diseño): pagos puntuales (98 %), NPS (7), satisfacción (4) y uso de la línea (30–33 %). Esto último refleja que, según Banxico, las tarjetas altas tienen límites más altos.
- **Por tarjeta**, el tier se ve en ingreso y gasto: Free (15.5k) < LikeU (24.3k) < Blanca / Oro (≈ 37k) < Gold (42.8k) < Platinum (≈ 54k) < World Elite (76.5k).
- **Free es la tarjeta más frágil:** tiene el menor e-commerce (37 %), la menor antigüedad (48 meses) y el mayor abandono mediano (0.14).
- **World Elite** tiene el mayor uso de la línea (37 %) y la menor activación de promociones (44 %): es el cliente menos sensible al precio.

## 8. Bolsas de oportunidad

Los cortes son percentiles de la base, salvo dos que vienen de la política v2 (señales de Banxico): **uso de la línea < 10 %** (cuenta casi inactiva) y **pagos puntuales < 80 %** (atrasos).

In [17]:
q = clientes.quantile(numeric_only=True, q=[.25, .75, .9])
FLAGS = {
    "Línea casi inactiva (uso < 10 %)": clientes.uso_tarjeta_pct < 10,
    "Atrasos (pagos puntuales < 80 %)": clientes.pagos_puntuales_pct < 80,
    "Baja activación de promos (p25)": clientes.activacion_promos_pct <= q.loc[.25, "activacion_promos_pct"],
    "Alto riesgo de abandono (p90)": clientes.abandono_riesgo >= q.loc[.9, "abandono_riesgo"],
    "Detractores NPS (0–6)": clientes.nps <= 6,
    "Baja adopción digital (p25 logins)": clientes.app_logins_mes <= q.loc[.25, "app_logins_mes"],
    "Alto gasto (p75)": clientes.gasto_mensual >= q.loc[.75, "gasto_mensual"],
}
FLAGS["Retención: alto riesgo + detractor"] = FLAGS["Alto riesgo de abandono (p90)"] & FLAGS["Detractores NPS (0–6)"]
bolsas = pd.DataFrame({k: f.groupby(clientes.segmento).mean().reindex(SEGMENTOS) * 100 for k, f in FLAGS.items()}).T
bolsas["Base"] = [f.mean() * 100 for f in FLAGS.values()]
bolsas["clientes"] = [int(f.sum()) for f in FLAGS.values()]
display(bolsas.rename_axis("% de clientes del segmento"))

fig = go.Figure()
for s in SEGMENTOS:
    fig.add_bar(y=bolsas.index, x=bolsas[s], name=s, orientation="h", marker_color=COLOR_SEG[s],
                hovertemplate="%{y}<br>" + s + ": %{x:.1f}%<extra></extra>")
fig.update_layout(barmode="group", height=520, yaxis_autorange="reversed", xaxis_title="% de clientes del segmento",
                  title="Bolsas de oportunidad por segmento", legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
fig.show()

# Abandono alto por tarjeta: ¿dónde se concentra el riesgo?
riesgo_tarjeta = (clientes.assign(alto=FLAGS["Alto riesgo de abandono (p90)"])
                  .groupby("tarjeta_santander").agg(clientes=("alto", "size"), alto_riesgo_pct=("alto", "mean"),
                                                    antiguedad_mediana=("antiguedad_meses", "median"))
                  .reindex(TARJETAS))
riesgo_tarjeta["alto_riesgo_pct"] *= 100
display(riesgo_tarjeta.round(1))

segmento,Mass,Select,Premier,Base,clientes
% de clientes del segmento,,,,,
Línea casi inactiva (uso < 10 %),15.76,14.42,11.06,15.24,1524
Atrasos (pagos puntuales < 80 %),5.30,5.65,4.68,5.31,531
Baja activación de promos (p25),23.29,30.67,34.04,25.09,2509
Alto riesgo de abandono (p90),11.57,5.58,1.99,10.05,1005
Detractores NPS (0–6),41.74,42.61,39.86,41.73,4173
Baja adopción digital (p25 logins),33.15,17.46,11.21,29.38,2938
Alto gasto (p75),4.82,100.00,100.00,25.00,2500
Retención: alto riesgo + detractor,7.65,3.89,1.42,6.68,668


,clientes,alto_riesgo_pct,antiguedad_mediana
tarjeta_santander,,,
Free,2520,14.00,48.00
LikeU,3389,10.60,64.00
Aeromexico Blanca,1243,9.80,66.00
Fiesta Rewards Oro,728,10.60,62.50
Santander Gold,1126,6.40,62.00
Santander Platinum,397,2.00,111.00
Fiesta Rewards Platinum,194,0.50,108.00
Aeromexico Platinum,276,3.30,101.00
World Elite,127,2.40,107.00


**Lectura — bolsas de oportunidad**

- **El riesgo de abandono se concentra en Mass:** 11.6 % de sus clientes está en el 10 % de mayor riesgo, contra 5.6 % en Select y 2.0 % en Premier. Por tarjeta, Free (14 %), LikeU y Fiesta Rewards Oro (10.6 %) y Aeroméxico Blanca (9.8 %) concentran el riesgo; las Platinum quedan en 0.5–3.3 %.
- **¿Qué explica el riesgo alto?** De los clientes en el p90 de riesgo, el 52 % tiene 12 meses o menos de antigüedad y el 42 % usa menos del 10 % de su línea (en la base, el 15 %). La política lo construye así, y apunta a dos palancas concretas: **onboarding** y **activación**.
- **Líneas casi inactivas** (uso < 10 %): 1,524 clientes (15.2 %), parecido al 17.5 % de cuentas inactivas de Banxico. Hay más en Mass (15.8 %) que en Premier (11.1 %).
- **Atrasos** (pagos < 80 %): 531 clientes (5.3 %), parejo entre segmentos. Es una bolsa de cobranza preventiva, no de un segmento.
- **Baja adopción digital:** se concentra en Mass (33 % vs 11 % en Premier) porque los logins suben con el gasto.
- **Baja activación de promociones:** sube con el segmento (23 % en Mass, 34 % en Premier). En Premier conviene ofrecer beneficios (viajes, experiencias) más que descuentos.
- **Retención prioritaria:** 668 clientes con riesgo alto que además son detractores, el 7.7 % de Mass.

## 9. Conclusiones

> 🤖 Conclusiones redactadas por una **IA (Claude, de Anthropic)** a partir de los resultados de arriba.

1. **La v2 combina el original con datos reales.** Ingreso, gasto, transacciones, logins, e-commerce y promociones quedan con la escala del original (KS ≤ 0.16). Niveles de tarjeta, totaleros, uso de la línea y morosidad coinciden con Banxico. Las 19 metas numéricas de la política se cumplen (celda de metas en `dataset_regeneration_v2.ipynb`).
2. **Lo que se aleja del original se aleja hacia la realidad.** Pagos puntuales, abandono, NPS y satisfacción dejan de ser uniformes: casi todos pagan a tiempo, la mayoría tiene riesgo de abandono bajo y la satisfacción es alta. Por eso el KS promedio (0.23) es mayor que en la v2 anterior, que copiaba la forma del original.
3. **Diferencias entre segmentos razonables.** Ninguna variable separa los segmentos por más de 1.7 σ, y uso de la línea, pagos y satisfacción son iguales entre segmentos. E-commerce (46 / 60 / 68 %) es la que más se nota fuera del bloque transaccional.
4. **Oportunidades que muestran los datos:**
   - **Onboarding de clientes nuevos:** el riesgo de abandono mediano baja de 0.35 a 0.12 entre el mes 1 y el mes 36, y la mitad del riesgo alto está en clientes de 12 meses o menos.
   - **Activación de líneas inactivas:** 1,524 clientes usan menos del 10 % de su línea, una señal fuerte de abandono.
   - **Retención en Mass,** sobre todo Free y LikeU: concentra el riesgo alto y los detractores (668 clientes en la bolsa de retención).
   - **Beneficios en lugar de descuentos en Premier y World Elite,** que activan menos promociones.
5. **Decisiones pendientes para el equipo:**
   - **Promociones vs ingreso:** tal como está escrita, la política no produce el efecto que dice. Propuesta: `−10·ln(ingreso / media del segmento)`, sin el `máx(0, ·)`.
   - **E-commerce entre segmentos:** si 46 / 60 / 68 % es demasiado, bajar el coeficiente del gasto de 12 a 5–6.
   - **Gasto ~ ingreso (ρ = 0.89):** si se quiere menos colinealidad, subir la σ del gasto (por ejemplo, de 0.25 a 0.4) y recalibrar los umbrales de segmento.
   - Confirmar si Select y Premier están intercambiados en el original (afecta la comparación por tier).